In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# Configuration
B, T, D = 2, 4, 8
EPS = 1e-5

# Sample input: [batch, sequence, model dimension]
x = torch.randn(B, T, D)

# RMSNorm implementation
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Normalize using root mean square
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        x = x / rms

        # Apply learnable scale
        return x * self.weight

# Create RMSNorm
norm = RMSNorm(D, eps=EPS)

# Forward pass
y = norm(x)

# Inspect shapes and values
print("Input shape: ", x.shape)
print("Output shape:", y.shape)
print("\nInput mean:", x.mean().item())
print("Input std: ", x.std().item())
print("Output mean:", y.mean().item())
print("Output std: ", y.std().item())

# Verify RMS is approximately 1 for each token
rms = torch.sqrt(torch.mean(y ** 2, dim=-1))
print("\nRMS per token:")
print(rms)

# Verify learnable parameter
print("\nLearnable weight shape:", norm.weight.shape)
print("Requires gradient:", norm.weight.requires_grad)

# Compare with PyTorch implementation if available
try:
    torch_rmsnorm = nn.RMSNorm(D, eps=EPS)
    torch_rmsnorm.weight.data.copy_(norm.weight.data)

    reference = torch_rmsnorm(x)

    print("\nMatches PyTorch RMSNorm:", torch.allclose(y, reference, atol=1e-6))
except AttributeError:
    print("\nPyTorch RMSNorm is not available in this version.")

Input shape:  torch.Size([2, 4, 8])
Output shape: torch.Size([2, 4, 8])

Input mean: 0.03282400593161583
Input std:  1.0574713945388794
Output mean: 0.0355924628674984
Output std:  1.0072613954544067

RMS per token:
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SqrtBackward0>)

Learnable weight shape: torch.Size([8])
Requires gradient: True

Matches PyTorch RMSNorm: True
